# Introduction to PyTorch
PyTorch is one of the most popular frameworks in the field of Deep Learning. Initially developed by Meta AI, and now it is a part of Linux Foundation. 

### 1) PyTorch Tensor
Tensors are the fundemental data structure in PyTorch, which is similar to an array or a matrix. It can support many mathematical operations and is the building block of neural networks. They can be created from Python list or NumPy arrays.


In [2]:
# import the PyTorch module
import torch

# define a tensor from a Python list
my_tensor = torch.tensor([[1, 2, 3], [4, 5, 6]])
print("Tensor:", my_tensor)
print("Shape:", my_tensor.shape)
print("Data type:", my_tensor.dtype)
print("Device:", my_tensor.device)

Tensor: tensor([[1, 2, 3],
        [4, 5, 6]])
Shape: torch.Size([2, 3])
Data type: torch.int64
Device: cpu


Mathematical operations can be done on two tensors as operands if they are compatible (same shape with aligned dimensions).

In [5]:
# define two operands
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])

# perform addition
c = a + b
print("Result of addition:", c)

# perform subtraction
d = a - b
print("Result of subtraction:", d)

# perform element-wise multiplication
e = a * b
print("Result of element-wise multiplication:", e)

# perform matrix multiplication (with matmul)
f = torch.matmul(a, b)
print("Result of matrix multiplication:", f)

# perform matrix multiplication (with @ operator)
g = a @ b
print("Result of matrix multiplication with @ operator:", g)

Result of addition: tensor([5, 7, 9])
Result of subtraction: tensor([-3, -3, -3])
Result of element-wise multiplication: tensor([ 4, 10, 18])
Result of matrix multiplication: tensor(32)
Result of matrix multiplication with @ operator: tensor(32)


### 2) Designing a Neural Network
A neural network consists of input, hidden and output layers where the input layer contains the dataset features, the output layer contains the predictions (classes) and the hidden layers lie in between. A layer is *fully connected* when each neuron links to all neurons in the previous layer. A neuron in a linear layer performs a linear operation using all neurons from the previous layer, hence has *N+1* parameters (*N* from inputs and *1* from the bias). More hidden layers means more parameters and higher model capacity. However, too many parameters can lead to long training times or overfitting.

- **NN with no hidden layers:**

In [ ]:
# define a nn with no hidden layers (fully connected, linear)
import torch.nn as nn

# create input tensor with 3 features
input_tensor = torch.tensor([1.0, 2.0, 3.0])

# define a linear layer with 3 input features and 2 output features
linear_layer = nn.Linear(in_features=3, out_features=2)

# pass the input tensor through the linear layer
output_tensor = linear_layer(input_tensor)

# print the output tensor, weights, and bias
print("Output tensor:", output_tensor)
print("Weights:", linear_layer.weight)
print("Bias:", linear_layer.bias)

Output tensor: tensor([-1.7973, -0.9706], grad_fn=<ViewBackward0>)
Weights: Parameter containing:
tensor([[-0.3875, -0.3943, -0.3769],
        [-0.4728,  0.3376, -0.5631]], requires_grad=True)
Bias: Parameter containing:
tensor([0.5096, 0.5163], requires_grad=True)


**NN with hidden layers:**

In [15]:
# create a fully connected nn with 3 linear layers and 2 hidden layers
model = nn.Sequential(
    nn.Linear(in_features=8, out_features=4),  # first hidden layer
    nn.Linear(in_features=4, out_features=2)   # output layer
)

input_tensor = torch.Tensor([[2, 3, 6, 7, 9, 3, 2, 1]])
output_tensor = model(input_tensor)
print("Output tensor:", output_tensor)

Output tensor: tensor([[0.6563, 1.2422]], grad_fn=<AddmmBackward0>)


**NN with activation functions:**
Activation functions add *non-linearity* to neural networks and make nns learn more complex relationships.
- Sigmoid: for binary classification
- Softmax: for multi-class classification


In [ ]:
# create a binary classification model with sigmoid activation function at output layer
# Note: A nn with a single layer followed by a sigmoid activation function is similar to a logistic regression model
model_sigmoid = nn.Sequential( 
    nn.Linear(in_features=8, out_features=4),  # first linear layer
    nn.Linear(in_features=4, out_features=1),   # second linear layer
    nn.Sigmoid()  # activation function
)

# create a 3 class multi-class classification model with softmax activation function at output layer
model_softmax = nn.Sequential( 
    nn.Linear(in_features=8, out_features=4),  # first linear layer
    nn.Linear(in_features=4, out_features=3),   # second linear layer
    nn.Softmax(dim=-1)  # activation function
)

**NN model parameters:**

In [18]:
import torch.optim as optim

# create a sequential model
model = nn.Sequential(
    nn.Linear(in_features=16, out_features=8),  # first linear layer
    nn.Linear(in_features=8, out_features=2)   # second linear layer
)

print("Model architecture:", model)
print("Model parameters:")

total_params = 0
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
    total_params += param.numel()

# model capacity = (16+1) * 8 + (8+1) * 2 = 154 learnable parameters
print(f"Total learnable parameters: {total_params}")

sample = torch.Tensor([[2, 3, 6, 7, 9, 3, 2, 1, 2, 3, 6, 7, 9, 3, 2, 1]])
target = torch.Tensor([[1, 9]])  # example target for binary classification
prediction = model(sample)

# create the optimizer
optimizer = optim.SGD(model.parameters(), lr=0.001)

# calculate the loss and gradients
criterion = nn.CrossEntropyLoss()
loss = criterion(prediction, target)
loss.backward()

# update the model's parameters using the optimizer
optimizer.step()

# access the gradients of the weight of the second linear layer (before backward pass)
grads1 = model[1].weight.grad
print("Gradient of the weight of the second layer:", grads1)

Model architecture: Sequential(
  (0): Linear(in_features=16, out_features=8, bias=True)
  (1): Linear(in_features=8, out_features=2, bias=True)
)
Model parameters:
0.weight: torch.Size([8, 16])
0.bias: torch.Size([8])
1.weight: torch.Size([2, 8])
1.bias: torch.Size([2])
Total learnable parameters: 154
Gradient of the weight of the second layer: tensor([[ -4.1883,   5.1305,  -5.7003,  -6.5260,   4.1386,  -3.4560,  15.6145,
          -1.6597],
        [  4.1883,  -5.1305,   5.7003,   6.5260,  -4.1386,   3.4560, -15.6145,
           1.6597]])


### 3) Training a Neural Network with PyTorch

**Loading Data:** 
Structuring data into a dataset is one of the first steps in training a PyTorch neural network. TensorDataset simplifies structuring data by converting NumPy arrays into a format PyTorch can use. After structuring the data, DataLoader class is essential for efficiently handling large datasets. It speeds up training, optimizes memory usage, and stabilizes gradient updates, making deep learning models more effective.

In [22]:
import torch
import pandas as pd
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

# create 'animals' as a pandas DataFrame containing the dataset
animals = pd.DataFrame({
    'animal_name': ['sparrow', 'eagle', 'cat', 'dog', 'lion', 'lizard'],
    'hair': [0, 0, 1, 1, 1, 0],
    'feathers': [1, 1, 0, 0, 0, 0],
    'eggs': [1, 1, 0, 0, 0, 1],
    'milk': [0, 0, 1, 1, 1, 0],
    'predator': [0, 1, 1, 0, 1, 1],
    'legs': [2, 2, 4, 4, 4, 4],
    'tail': [1, 1, 1, 1, 1, 1],
    'type': [0, 0, 1, 1, 1, 2]  # bird (0), mammal (1), reptile (2)
})

# extract features and target variable from the DataFrame
X = animals.iloc[:, 1:-1].to_numpy()  
y = animals.iloc[:, -1].to_numpy()

# create a tensor dataset
# CrossEntropyLoss expects LongTensor (int64) for labels, and float32 for features
dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))

# print the first sample
input_sample, label_sample = dataset[0]
print('Input sample:', input_sample)
print('Label sample:', label_sample)

# create a DataLoader
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# iterate over the dataloader
for batch_inputs, batch_labels in dataloader:
    print('batch_inputs:', batch_inputs)
    print('batch_labels:', batch_labels)

Input sample: tensor([0., 1., 1., 0., 0., 2., 1.])
Label sample: tensor(0)
batch_inputs: tensor([[1., 0., 0., 1., 1., 4., 1.],
        [0., 1., 1., 0., 1., 2., 1.]])
batch_labels: tensor([1, 0])
batch_inputs: tensor([[1., 0., 0., 1., 0., 4., 1.],
        [1., 0., 0., 1., 1., 4., 1.]])
batch_labels: tensor([1, 1])
batch_inputs: tensor([[0., 1., 1., 0., 0., 2., 1.],
        [0., 0., 1., 0., 1., 4., 1.]])
batch_labels: tensor([0, 2])


**Training Loop:** Looping through the entire dataset once is called an epoch. In practice, we often loop through the dataset multiple times (multiple epochs) to allow the model to learn better representations of the data. In ```scikit-learn```, the training loop is wrapped in the ```.fit()``` method, while in PyTorch, it's set up manually. While this adds flexibility, it requires a custom implementation.

In [32]:
# define a simple model for multi-class classification
model = nn.Sequential(
    nn.Linear(in_features=7, out_features=8),  # first linear layer
    nn.ReLU(),  # activation function
    nn.Linear(in_features=8, out_features=3),   # second linear layer
)

# define the loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

num_epochs = 30

# loop over the number of epochs and the dataloader
for i in range(num_epochs):
  for data in dataloader:
    # set the gradients to zero
    optimizer.zero_grad()

    # run a forward pass
    feature, target = data
    prediction = model(feature)

    # compute the loss
    loss = criterion(prediction, target)

    # compute the gradients
    loss.backward()

    # update the model parameters
    optimizer.step()

# show evaluation results
model.eval()
with torch.no_grad():
    for feature, target in dataloader:
        prediction = model(feature)
        predicted_classes = prediction.argmax(dim=1)
        for p, t in zip(predicted_classes, target):
            print(f'Ground truth type: {t.item()}. Predicted type: {p.item()}.')

Ground truth type: 0. Predicted type: 0.
Ground truth type: 2. Predicted type: 2.
Ground truth type: 1. Predicted type: 1.
Ground truth type: 0. Predicted type: 0.
Ground truth type: 1. Predicted type: 1.
Ground truth type: 1. Predicted type: 1.


The *learning rate* (which is set as 0.01 above) controls the step size and setting a rate too high causes poor performance, while setting it too low results in slower training. Typical range for *learning rate* is 0.01 and 0.0001. On the other hand, *momentum* controls the inertia and helps escaping the local minimum. If it is set too small, optimizer gets stuck. The typical range for *momentum* is 0.85 to 0.99.